In [9]:
import torch
import torch.nn as nn
import random as rand

from hidden_hand import HiddenHandEnv, action_name

In [5]:
env = HiddenHandEnv(seed=10)
obs = env.reset()

# Extract the board tokens
board_tokens = obs["tokens"]["board_tokens"]

# Convert to a PyTorch tensor
board_tensor = torch.tensor(board_tokens, dtype=torch.float32)

print("Original board tokens:")
print(board_tokens)

print("\nBoard tensor:")
print(board_tensor)

print("\nTensor shape:", board_tensor.shape)
print("Tensor data type:", board_tensor.dtype)

Original board tokens:
[[1, 0, 0, -99], [1, 1, 1, 3], [1, 2, 0, -99], [1, 3, 0, -99], [0, 0, 0, -99], [0, 1, 0, -99], [0, 2, 0, -99], [0, 3, 0, -99]]

Board tensor:
tensor([[  1.,   0.,   0., -99.],
        [  1.,   1.,   1.,   3.],
        [  1.,   2.,   0., -99.],
        [  1.,   3.,   0., -99.],
        [  0.,   0.,   0., -99.],
        [  0.,   1.,   0., -99.],
        [  0.,   2.,   0., -99.],
        [  0.,   3.,   0., -99.]])

Tensor shape: torch.Size([8, 4])
Tensor data type: torch.float32


Convert the rest of the tokens to tensors

In [6]:
tokens = obs["tokens"]

discard_tensor = torch.tensor(
    tokens["discard_token"],
    dtype=torch.float32
)

global_tensor = torch.tensor(
    tokens["global_token"],
    dtype=torch.float32
)

decision_tensor = torch.tensor(
    tokens["decision_token"],
    dtype=torch.float32
)

action_tensor = torch.tensor(
    tokens["action_tokens"],
    dtype=torch.float32
)

print("Discard:", discard_tensor.shape)
print("Global:", global_tensor.shape)
print("Decision:", decision_tensor.shape)
print("Actions:", action_tensor.shape)

Discard: torch.Size([2])
Global: torch.Size([6])
Decision: torch.Size([2])
Actions: torch.Size([2, 6])


In [11]:
# Make a copy of the original tensor
clean_board = board_tensor.clone();

# Loop through each card
for card in clean_board:

    # Check if the card's value is unknown
    if card[3] == -99:

        # Replace the unknown value with zero
        card[3] = 0.0

print("Original board:")
print(board_tensor)

print("\nCleaned board:")
print(clean_board)
print("\n")

clean_discard = discard_tensor.clone()

# Record whether a discarded card exists
discard_known = 1.0

if clean_discard[0] == -99:
    discard_known = 0.0
    clean_discard[0] = 0.0

print("Clean discard:", clean_discard)
print("Discard known:", discard_known)
print("\n")

# Clean Decision

clean_decision = decision_tensor.clone()

if clean_decision[1] == -99:
    clean_decision[1] = 0.0

print("Clean decision:", clean_decision)
print("\n")

#Global should already be clean
clean_global = global_tensor.clone()

#Clean Actions

clean_actions = action_tensor.clone()

# Create a mask that records which fields are available
action_value_mask = torch.ones_like(clean_actions)

for i in range(len(clean_actions)):
    for j in range(len(clean_actions[i])):

        if clean_actions[i][j] == -127:
            clean_actions[i][j] = 0.0
            action_value_mask[i][j] = 0.0

print("Clean actions:")
print(clean_actions)

print("\nAction value mask:")
print(action_value_mask)

Original board:
tensor([[  1.,   0.,   0., -99.],
        [  1.,   1.,   1.,   3.],
        [  1.,   2.,   0., -99.],
        [  1.,   3.,   0., -99.],
        [  0.,   0.,   0., -99.],
        [  0.,   1.,   0., -99.],
        [  0.,   2.,   0., -99.],
        [  0.,   3.,   0., -99.]])

Cleaned board:
tensor([[1., 0., 0., 0.],
        [1., 1., 1., 3.],
        [1., 2., 0., 0.],
        [1., 3., 0., 0.],
        [0., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 2., 0., 0.],
        [0., 3., 0., 0.]])


Clean discard: tensor([0., 0.])
Discard known: 0.0


Clean decision: tensor([1., 7.])


Clean actions:
tensor([[0., 2., 0., 0., 1., 0.],
        [1., 2., 0., 1., 0., 0.]])

Action value mask:
tensor([[1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 0.]])


We need to pad the action history as its size can change and this will not work for the NN

In [12]:
# Maximum number of actions we want to store
MAX_HISTORY = 20

# Each action contains 6 features
ACTION_FEATURES = 6

# Create an empty tensor with 20 rows and 6 columns
padded_actions = torch.zeros(
    MAX_HISTORY,
    ACTION_FEATURES,
    dtype=torch.float32
)

# Create a mask containing 20 zeros
history_mask = torch.zeros(
    MAX_HISTORY,
    dtype=torch.float32
)

# Determine how many actions have actually occurred
num_actions = min(len(clean_actions), MAX_HISTORY)

# Keep only the most recent 20 actions, if necessary
recent_actions = clean_actions[-MAX_HISTORY:]

# Copy real actions into the padded tensor
for i in range(num_actions):
    padded_actions[i] = recent_actions[i]

    # Mark this row as containing a real action
    history_mask[i] = 1.0

print("Padded actions:")
print(padded_actions)

print("\nHistory mask:")
print(history_mask)

print("\nPadded actions shape:", padded_actions.shape)
print("History mask shape:", history_mask.shape)



Padded actions:
tensor([[0., 2., 0., 0., 1., 0.],
        [1., 2., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

History mask:
tensor([1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])

Padded actions shape: torch.Size([20, 6])
History mask shape: torch.Size([20])


Double checking proper dimensions

In [14]:
print(padded_actions.shape)     #([20, 6])
print(history_mask.shape)       #([20])
print(padded_value_mask.shape)  #([20, 6])

torch.Size([20, 6])
torch.Size([20])
torch.Size([20, 6])


We want to normalize these values.

- Card values range from 0 to 9.
- Deck size ranges from 0 to 20.
- Game progress ranges from 0 to 16 steps.
- Player IDs and visibility indicators are already 0 or 1.

We will normalize from [0, 1]

In [17]:

# Normalizing Board first

# Copy our cleaned board tensor
normalized_board = clean_board.clone();

for card in normalized_board:
    # Normalize the card position (from 0 to 3 originally)
    card[1] = card[1] / 3
    # Normalize the card value (From 0 to 9 originally)
    card[3] = card[3] / 9

# Normalize Discord

normalized_discard = clean_discard.clone();

# Normalize the top discarded card
normalized_discard[0] = normalized_discard[0] / 9.0

# Normalize the number of discarded cards.
normalized_discard[1] = normalized_discard[1] / 16.0

## Normalize Global

normalized_global = clean_global.clone()

# Deck size ranges from 0 to 20
normalized_global[0] = normalized_global[0] / 20.0

# Maximum number of player actions is 16
normalized_global[1] = normalized_global[1] / 16.0

# Current player is already 0 or 1
# Leave index 2 unchanged

# Phase ranges from 0 to 2
normalized_global[3] = normalized_global[3] / 2.0

# Phase bucket ranges from 0 to 2
normalized_global[4] = normalized_global[4] / 2.0

# Normalize decision

normalized_decision = clean_decision.clone()

# Normalize the drawn card value
normalized_decision[1] = normalized_decision[1] / 9.0

## Normalize Action history

# Copy our padded action history
normalized_actions = padded_actions.clone()

for i in range(len(normalized_actions)):

    # Skip padded rows
    if history_mask[i] == 0:
        continue

    action = normalized_actions[i]

    # Actor is already 0 or 1
    # Leave index 0 unchanged

    # Action type ranges from 1 to 4
    action[1] = action[1] / 4.0

    # Source position ranges from 0 to 3
    action[2] = action[2] / 3.0

    # Target position ranges from 0 to 3
    action[3] = action[3] / 3.0

    # Tell value is already 0 or 1
    # Leave index 4 unchanged

    # Discarded card value ranges from 0 to 9
    action[5] = action[5] / 9.0

print("\nNormalized board:")
print(normalized_board)

print("\nNormalized discard:")
print(normalized_discard)
print("Discard known:", discard_known)

print("\nNormalized global:")
print(normalized_global)

print("\nNormalized decision:")
print(normalized_decision)

print("\nNormalized action history:")
print(normalized_actions)


Normalized board:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.3333, 1.0000, 0.3333],
        [1.0000, 0.6667, 0.0000, 0.0000],
        [1.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3333, 0.0000, 0.0000],
        [0.0000, 0.6667, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000]])

Normalized discard:
tensor([0., 0.])
Discard known: 0.0

Normalized global:
tensor([0.9500, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000])

Normalized decision:
tensor([1.0000, 0.7778])
